In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import psycopg2
from sqlalchemy import create_engine, text
from datasets import load_dataset
from dotenv import load_dotenv
import os
import json
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print("✅ All imports loaded")
print(f"   Transformers and datasets ready")

c:\Users\saich\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports loaded
   Transformers and datasets ready


In [2]:
# Cell 2 — Load AG News from Hugging Face
print("Loading AG News dataset from Hugging Face...")
print("This may take 2-3 minutes on first load...\n")

dataset = load_dataset("ag_news")

print(f"✅ Dataset loaded successfully")
print(f"\nDataset structure:")
print(dataset)


Loading AG News dataset from Hugging Face...
This may take 2-3 minutes on first load...



Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 300959.34 examples/s]

✅ Dataset loaded successfully

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [3]:
# Cell 3 — Explore structure
train_data = dataset['train']
test_data  = dataset['test']

# Label mapping
label_names = {
    0: 'World',
    1: 'Sports',
    2: 'Business',
    3: 'Sci/Tech'
}

print("=== AG NEWS DATASET OVERVIEW ===\n")
print(f"Training samples: {len(train_data):,}")
print(f"Test samples:     {len(test_data):,}")
print(f"Features:         {train_data.features}")
print(f"\nLabel mapping:")
for k, v in label_names.items():
    print(f"  {k} → {v}")

# Show sample records
print(f"\n=== SAMPLE RECORDS ===")
for i in range(3):
    sample = train_data[i]
    label  = label_names[sample['label']]
    text   = sample['text'][:100]
    print(f"\nSample {i+1}:")
    print(f"  Label: {label}")
    print(f"  Text:  {text}...")

=== AG NEWS DATASET OVERVIEW ===

Training samples: 120,000
Test samples:     7,600
Features:         {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}

Label mapping:
  0 → World
  1 → Sports
  2 → Business
  3 → Sci/Tech

=== SAMPLE RECORDS ===

Sample 1:
  Label: Business
  Text:  Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b...

Sample 2:
  Label: Business
  Text:  Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,...

Sample 3:
  Label: Business
  Text:  Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about th...


In [5]:
# Cell 4 — Convert to pandas
df_train = pd.DataFrame({
    'text':       train_data['text'],
    'label':      train_data['label'],
    'label_name': [label_names[l] for l
                   in train_data['label']]
})

df_test = pd.DataFrame({
    'text':       test_data['text'],
    'label':      test_data['label'],
    'label_name': [label_names[l] for l
                   in test_data['label']]
})

print(f"Train DataFrame shape: {df_train.shape}")
print(f"Test DataFrame shape:  {df_test.shape}")
print(f"\nFirst 3 rows:")
print(df_train.head(3))
print(f"\nData types:")
print(df_train.dtypes)

Train DataFrame shape: (120000, 3)
Test DataFrame shape:  (7600, 3)

First 3 rows:
                                                text  label label_name
0  Wall St. Bears Claw Back Into the Black (Reute...      2   Business
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2   Business
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2   Business

Data types:
text          object
label          int64
label_name    object
dtype: object


In [6]:
# Cell 5 — Data quality audit
print("=== DATA QUALITY AUDIT ===\n")

# Null check
print(f"Null values in train:")
print(df_train.isnull().sum())

# Duplicate check
dupes = df_train.duplicated(subset=['text']).sum()
print(f"\nDuplicate texts: {dupes}")

# Text length stats
df_train['text_length'] = df_train['text'].str.len()
df_train['word_count']  = df_train['text'].str.split().str.len()

print(f"\nText length stats:")
print(df_train[['text_length', 'word_count']].describe().round(2))

# Class distribution
print(f"\nClass distribution:")
dist = df_train['label_name'].value_counts()
for label, count in dist.items():
    pct = count / len(df_train) * 100
    print(f"  {label:<12}: {count:,} ({pct:.1f}%)")

=== DATA QUALITY AUDIT ===

Null values in train:
text          0
label         0
label_name    0
dtype: int64

Duplicate texts: 0

Text length stats:
       text_length  word_count
count    120000.00   120000.00
mean        236.48       37.85
std          66.51       10.09
min         100.00        8.00
25%         196.00       32.00
50%         232.00       37.00
75%         266.00       43.00
max        1012.00      177.00

Class distribution:
  Business    : 30,000 (25.0%)
  Sci/Tech    : 30,000 (25.0%)
  Sports      : 30,000 (25.0%)
  World       : 30,000 (25.0%)


In [7]:
# Cell 6 — Upload to AWS S3
print("Uploading dataset to AWS S3...")

s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv(
        'AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv(
        'AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_DEFAULT_REGION')
)

bucket = os.getenv('S3_BUCKET_DATA')

# Save locally first then upload
os.makedirs('../data/raw', exist_ok=True)

# Save as CSV
train_path = '../data/raw/ag_news_train.csv'
test_path  = '../data/raw/ag_news_test.csv'

df_train.to_csv(train_path, index=False)
df_test.to_csv(test_path,  index=False)

print(f"✅ Saved locally")

# Upload to S3
for local_path, s3_key in [
    (train_path, 'raw/ag_news_train.csv'),
    (test_path,  'raw/ag_news_test.csv')
]:
    s3_client.upload_file(
        local_path,
        bucket,
        s3_key
    )
    print(f"✅ Uploaded to s3://{bucket}/{s3_key}")

# Verify uploads
response = s3_client.list_objects_v2(
    Bucket=bucket, Prefix='raw/')
print(f"\n=== S3 BUCKET CONTENTS ===")
for obj in response.get('Contents', []):
    size_mb = obj['Size'] / (1024*1024)
    print(f"  {obj['Key']:<35} "
          f"{size_mb:.2f} MB")

Uploading dataset to AWS S3...
✅ Saved locally
✅ Uploaded to s3://document-ai-portfolio-data/raw/ag_news_train.csv
✅ Uploaded to s3://document-ai-portfolio-data/raw/ag_news_test.csv

=== S3 BUCKET CONTENTS ===
  raw/ag_news_test.csv                1.80 MB
  raw/ag_news_train.csv               29.39 MB


In [9]:
# Cell 7 — Create database tables in RDS
from sqlalchemy import text

print("Setting up RDS schema...")

connection_string = (
    f"postgresql://"
    f"{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"{os.getenv('POSTGRES_HOST')}:"
    f"{os.getenv('POSTGRES_PORT')}/"
    f"{os.getenv('POSTGRES_DB')}"
)

engine = create_engine(connection_string)

# Create tables
with engine.connect() as conn:

    # Documents table — stores metadata
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS documents (
            id              SERIAL PRIMARY KEY,
            text            TEXT NOT NULL,
            true_label      INTEGER,
            label_name      VARCHAR(50),
            predicted_label INTEGER,
            predicted_name  VARCHAR(50),
            confidence      FLOAT,
            sentiment       VARCHAR(20),
            sentiment_score FLOAT,
            processed_at    TIMESTAMP
                DEFAULT CURRENT_TIMESTAMP,
            source          VARCHAR(100)
                DEFAULT 'ag_news',
            split           VARCHAR(10)
        )
    """))

    # Predictions log table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS
        prediction_logs (
            id              SERIAL PRIMARY KEY,
            document_id     INTEGER
                REFERENCES documents(id),
            model_version   VARCHAR(50),
            prediction      INTEGER,
            confidence      FLOAT,
            inference_time_ms FLOAT,
            created_at      TIMESTAMP
                DEFAULT CURRENT_TIMESTAMP
        )
    """))

    # Model metrics table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS
        model_metrics (
            id              SERIAL PRIMARY KEY,
            model_name      VARCHAR(100),
            experiment_id   VARCHAR(100),
            run_id          VARCHAR(100),
            accuracy        FLOAT,
            f1_score        FLOAT,
            precision_score FLOAT,
            recall_score    FLOAT,
            eval_samples    INTEGER,
            created_at      TIMESTAMP
                DEFAULT CURRENT_TIMESTAMP
        )
    """))

    conn.commit()

print("✅ Tables created in RDS:")
print("   documents       — stores text + predictions")
print("   prediction_logs — tracks every inference")
print("   model_metrics   — stores evaluation results")

Setting up RDS schema...
✅ Tables created in RDS:
   documents       — stores text + predictions
   prediction_logs — tracks every inference
   model_metrics   — stores evaluation results


In [10]:
# Cell 8 — Insert sample records into RDS
print("Loading sample data into RDS...")

# Use 1000 samples for now
# Full dataset loaded during Airflow pipeline
sample_df = df_train.sample(
    1000, random_state=42).reset_index(drop=True)

# Prepare for insertion
sample_df_rds = sample_df[[
    'text', 'label', 'label_name']].copy()
sample_df_rds.columns = [
    'text', 'true_label', 'label_name']
sample_df_rds['split'] = 'train'

# Insert into RDS
sample_df_rds.to_sql(
    'documents',
    engine,
    if_exists='append',
    index=False,
    method='multi',
    chunksize=100
)

print(f"✅ Inserted {len(sample_df_rds):,} "
      f"records into RDS")

# Verify with SQL query
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            label_name,
            COUNT(*) as count,
            ROUND(COUNT(*) * 100.0 /
                SUM(COUNT(*)) OVER(), 2)
                as percentage
        FROM documents
        GROUP BY label_name
        ORDER BY count DESC
    """))

    print("\n=== RDS DOCUMENTS TABLE ===")
    print(f"{'Category':<15} {'Count':>8} "
          f"{'Percentage':>12}")
    print("-" * 38)
    for row in result:
        print(f"{row[0]:<15} {row[1]:>8,} "
              f"{row[2]:>11.1f}%")

Loading sample data into RDS...
✅ Inserted 1,000 records into RDS

=== RDS DOCUMENTS TABLE ===
Category           Count   Percentage
--------------------------------------
Sports               277        27.7%
Sci/Tech             261        26.1%
World                236        23.6%
Business             226        22.6%


In [11]:
# Cell 9 — Save summary
os.makedirs('../logs', exist_ok=True)

summary = {
    'dataset':          'AG News',
    'source':           'Hugging Face datasets',
    'train_samples':    int(len(df_train)),
    'test_samples':     int(len(df_test)),
    'num_classes':      4,
    'classes':          list(label_names.values()),
    'avg_text_length':  round(float(
        df_train['text_length'].mean()), 2),
    'avg_word_count':   round(float(
        df_train['word_count'].mean()), 2),
    'class_balance':    'Perfectly balanced '
                        '(30000 per class)',
    's3_location':      f"s3://{bucket}/raw/",
    'rds_table':        'documents',
    'rds_records':      1000
}

with open('../logs/dataset_summary.json', 'w') as f:
    json.dump(summary, f, indent=4)

print("✅ Summary saved to logs/dataset_summary.json")
print(json.dumps(summary, indent=4))

✅ Summary saved to logs/dataset_summary.json
{
    "dataset": "AG News",
    "source": "Hugging Face datasets",
    "train_samples": 120000,
    "test_samples": 7600,
    "num_classes": 4,
    "classes": [
        "World",
        "Sports",
        "Business",
        "Sci/Tech"
    ],
    "avg_text_length": 236.48,
    "avg_word_count": 37.85,
    "class_balance": "Perfectly balanced (30000 per class)",
    "s3_location": "s3://document-ai-portfolio-data/raw/",
    "rds_table": "documents",
    "rds_records": 1000
}
